In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_excel('../data/Telco_customer_churn.xlsx')
columns_to_drop = ['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 
                   'Lat Long', 'Latitude', 'Longitude', 'Churn Value', 'Churn Score', 
                   'CLTV', 'Churn Reason', 'TotalCharges']
cols_to_drop = [col for col in columns_to_drop if col in df.columns]
df.drop(columns=cols_to_drop, inplace=True)
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df.dropna(subset=['Total Charges'], inplace=True)

X = df.drop(columns=['Churn Label'])
y = df['Churn Label'].map({'Yes': 1, 'No': 0})
X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Number of features (inputs): {X_train_tensor.shape[1]}")
print("Data is loaded, scaled, and batched into Tensors")

Number of features (inputs): 30
Data is loaded, scaled, and batched into Tensors!


In [2]:
class ChurnMLP(nn.Module):
    def __init__(self, input_size):
        super(ChurnMLP, self).__init__()
        
        self.layer1 = nn.Linear(input_size, 64)  
        self.relu1 = nn.ReLU()                   
        
        self.layer2 = nn.Linear(64, 32)          
        self.relu2 = nn.ReLU()                  
        
        self.output_layer = nn.Linear(32, 1)     

    def forward(self, x):
        out = self.layer1(x)
        out = self.relu1(out)
        out = self.layer2(out)
        out = self.relu2(out)
        out = self.output_layer(out)
        return out

input_features = X_train_tensor.shape[1]
model = ChurnMLP(input_size=input_features)

print("Neural Network Architecture Defined:")
print(model)

Neural Network Architecture Defined:
ChurnMLP(
  (layer1): Linear(in_features=30, out_features=64, bias=True)
  (relu1): ReLU()
  (layer2): Linear(in_features=64, out_features=32, bias=True)
  (relu2): ReLU()
  (output_layer): Linear(in_features=32, out_features=1, bias=True)
)


In [3]:
import copy

epochs = 150              
patience = 10             
best_loss = float('inf')
patience_counter = 0
best_model_weights = None

criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Starting Training Loop with Early Stopping...\n")

for epoch in range(epochs):
    model.train() 
    train_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                 
        outputs = model(X_batch)              
        loss = criterion(outputs, y_batch)    
        loss.backward()                       
        optimizer.step()                      
        
        train_loss += loss.item() * X_batch.size(0)
        
    train_loss /= len(train_loader.dataset)
    
    model.eval()  
    val_loss = 0.0
    with torch.no_grad(): 
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * X_batch.size(0)
            
    val_loss /= len(test_loader.dataset)
    
    if val_loss < best_loss:
        best_loss = val_loss
        best_model_weights = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        
    if (epoch + 1) % 10 == 0 or patience_counter >= patience:
        print(f"Epoch {epoch+1:03d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        
    if patience_counter >= patience:
        print(f"\nEarly stopping triggered at epoch {epoch+1}!")
        break

model.load_state_dict(best_model_weights)
print("Best model weights restored. Training complete")

Starting Training Loop with Early Stopping...

Epoch 010/150 | Train Loss: 0.3784 | Val Loss: 0.4204
Epoch 020/150 | Train Loss: 0.3602 | Val Loss: 0.4260

Early stopping triggered at epoch 20!
Best model weights restored. Training complete!


In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import mlflow

model.eval()
y_true = []
y_pred_probs = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        probs = torch.sigmoid(outputs)
        y_pred_probs.extend(probs.numpy())
        y_true.extend(y_batch.numpy())

y_true = [y[0] for y in y_true]
y_pred_probs = [p[0] for p in y_pred_probs]
y_pred = [1 if p >= 0.5 else 0 for p in y_pred_probs]

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_pred_probs)

print("--- PyTorch MLP Performance ---")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f} (Crucial for catching churners!)")
print(f"F1-Score:  {f1:.4f}")
print(f"AUC-ROC:   {roc_auc:.4f}\n")

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
print("--- Análise de Custo (Trade-off) ---")
print(f"Falsos Positivos (FP): {fp} -> O modelo achou que iam cancelar, mas ficaram.")
print(f"Falsos Negativos (FN): {fn} -> O modelo achou que iam ficar, mas cancelaram.")
print("\nBusiness Insight: Em retenção de clientes, Falsos Negativos (perder o cliente) quase sempre custam muito mais caro do que Falsos Positivos (dar um desconto desnecessário).")

mlflow.set_experiment("Telco_Churn_PyTorch")
with mlflow.start_run(run_name="PyTorch_MLP_Final"):
    mlflow.log_param("model_type", "PyTorch_MLP")
    mlflow.log_param("optimizer", "Adam")
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    print("\nExperiment fully logged to MLflow")

--- PyTorch MLP Performance ---
Accuracy:  0.8017
Precision: 0.6426
Recall:    0.5722 (Crucial for catching churners!)
F1-Score:  0.6054
AUC-ROC:   0.8428

--- Análise de Custo (Trade-off) ---
Falsos Positivos (FP): 119 -> O modelo achou que iam cancelar, mas ficaram.
Falsos Negativos (FN): 160 -> O modelo achou que iam ficar, mas cancelaram.

Business Insight: Em retenção de clientes, Falsos Negativos (perder o cliente) quase sempre custam muito mais caro do que Falsos Positivos (dar um desconto desnecessário).


2026/06/03 17:15:16 INFO mlflow.tracking.fluent: Experiment with name 'Telco_Churn_PyTorch' does not exist. Creating a new experiment.



Experiment fully logged to MLflow!


In [5]:
import joblib
import torch

# 1. Save the Scikit-Learn Scaler
joblib.dump(scaler, '../models/scaler.pkl')

# 2. Save the PyTorch Model Weights
torch.save(model.state_dict(), '../models/churn_mlp.pth')

print("Artifacts successfully saved to the models/ folder")

Artifacts successfully saved to the models/ folder!
